# NQ range hit rates — split by break direction

Fixes two things from the last version:

1. **Every stat now comes three ways** — pooled, `_up` (ranges that broke the
   HIGH) and `_dn` (ranges that broke the LOW). Previously the continuation and
   reversion columns mixed both directions, so you couldn't ask what happens
   after a downside break.
2. **Lean is measured against the real base rate, not 50%.** NQ breaks high
   52.2% of the time overall, so a window at 52% is average, not bullish. The
   `lean_pp` column is signed: positive leans up, negative leans down.

Put this next to `nq_15y_1min.csv` and run top to bottom.

## Cell 1 — engine

In [1]:
import os, sys, math, warnings
import numpy as np
import pandas as pd

# ---- data.py -----------------------------------------------------
NY = "America/New_York"
OHLC = ["Open", "High", "Low", "Close"]

# Verified per-file source timezones (see detect_timezone to check a new file).
KNOWN_TZ = {
    "nq_15y_1min.csv": NY,
    "GC_15y_1min.csv": "UTC",
}


def _read_raw(path, usecols=None):
    """Read the Date,Time,O,H,L,C[,Volume] export as-is (no tz handling)."""
    cols = usecols or ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]
    head = pd.read_csv(path, nrows=0)
    cols = [c for c in cols if c in head.columns]
    dtypes = {c: "float64" for c in cols if c in OHLC + ["Volume"]}
    return pd.read_csv(path, usecols=cols, dtype={**dtypes, "Date": str, "Time": str})


def detect_timezone(path, verbose=True):
    """
    Infer the file's timezone from the CME daily maintenance break (17:00-18:00 ET).

    Returns "America/New_York" if the emptiest hour sits at 17:00 in both winter
    and summer, "UTC" if it shifts by one hour between them (tracking the ET-UTC
    offset), else None - in which case inspect it by hand before trusting it.
    """
    df = _read_raw(path, usecols=["Date", "Time"])
    hour = pd.to_datetime(df["Time"], format="%H:%M:%S").dt.hour
    month = df["Date"].str.slice(5, 7).astype(int)

    def emptiest(mask):
        sub = hour[mask]
        if len(sub) < 1000:
            return None
        return int(sub.value_counts().reindex(range(24), fill_value=0).idxmin())

    winter = emptiest(month.isin([12, 1, 2]))
    summer = emptiest(month.isin([6, 7, 8]))

    if winter is None or summer is None:
        tz = None
    elif winter == summer == 17:
        tz = NY
    elif (winter - summer) % 24 == 1:
        tz = "UTC"
    else:
        tz = None

    if verbose:
        print(f"{os.path.basename(path)}: emptiest hour winter={winter}:00 "
              f"summer={summer}:00 -> {tz or 'UNRECOGNISED - inspect manually'}")
    return tz


def load_bars(path, tz=None, start=None, end=None, symbol=None):
    """
    Load 1-minute bars as the engine's expected frame.

    tz    : source timezone of the timestamps. None = look up KNOWN_TZ by
            filename, else auto-detect. Pass explicitly for a new file.
    start/end : optional ET date strings, e.g. "2024-01-01". Filtering happens
            AFTER conversion, so the bounds mean what you'd expect.

    Returns columns Datetime (tz-aware ET), Open, High, Low, Close [, Volume],
    sorted, de-duplicated on Datetime.
    """
    base = os.path.basename(path)
    if tz is None:
        tz = KNOWN_TZ.get(base) or detect_timezone(path, verbose=False)
    if tz is None:
        raise ValueError(
            f"could not determine the timezone of {base}. Run "
            f"detect_timezone() and pass tz= explicitly - guessing would "
            f"silently shift every session boundary.")

    df = _read_raw(path)
    naive = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y-%m-%d %H:%M:%S")

    if tz == NY:
        # Wall-clock ET. The spring-forward hour does not exist and the
        # fall-back hour is ambiguous; the 17:00-18:00 CME break means the
        # 02:00 DST switch sits inside live trading, so resolve rather than raise.
        dt = naive.dt.tz_localize(NY, ambiguous=True, nonexistent="shift_forward")
    else:
        dt = naive.dt.tz_localize(tz).dt.tz_convert(NY)

    out = pd.DataFrame({"Datetime": dt})
    for c in OHLC:
        out[c] = df[c].to_numpy()
    if "Volume" in df.columns:
        out["Volume"] = df["Volume"].to_numpy()

    out = (out.dropna(subset=["Datetime"] + OHLC)
              .drop_duplicates("Datetime")
              .sort_values("Datetime", kind="mergesort")
              .reset_index(drop=True))

    if start is not None:
        out = out[out["Datetime"] >= pd.Timestamp(start, tz=NY)]
    if end is not None:
        out = out[out["Datetime"] <= pd.Timestamp(end, tz=NY)]
    out = out.reset_index(drop=True)
    out.attrs["symbol"] = symbol or base.split("_")[0].upper()
    out.attrs["source_tz"] = tz
    return out


def audit_bars(df, label=""):
    """
    Sanity report before you trust a backtest on this frame. Checks the things
    that quietly break range logic: session coverage, duplicate or missing
    minutes, zero/negative prices, and OHLC consistency.
    """
    d = df["Datetime"]
    sess = (d - pd.Timedelta(hours=18)).dt.date      # 18:00 ET session anchor
    bad_ohlc = (
        (df["High"] < df["Low"])
        | (df["High"] < df[["Open", "Close"]].max(axis=1))
        | (df["Low"] > df[["Open", "Close"]].min(axis=1))
    ).sum()
    rows = {
        "bars": len(df),
        "first bar (ET)": str(d.iloc[0]),
        "last bar (ET)": str(d.iloc[-1]),
        "sessions": sess.nunique(),
        "duplicate timestamps": int(df["Datetime"].duplicated().sum()),
        "non-positive prices": int((df[OHLC] <= 0).any(axis=1).sum()),
        "OHLC inconsistencies": int(bad_ohlc),
        "median bars/session": int(sess.value_counts().median()),
        "sessions < 600 bars": int((sess.value_counts() < 600).sum()),
    }
    width = max(len(k) for k in rows)
    print(f"--- bar audit {label or df.attrs.get('symbol', '')} ---")
    for k, v in rows.items():
        print(f"  {k:<{width}} : {v}")
    return rows

# ---- ranges.py ---------------------------------------------------
NY = "America/New_York"
ANCHOR_HOUR = 18

# The MTF notebook's open-position buckets, kept identical so results line up.
OP_EDGES = [-np.inf, 0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0, np.inf]
OP_LABELS = ["below pLow", "0.0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5",
             "0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0", "pHigh or above"]


def _session_date(dt: pd.Series) -> pd.Series:
    """18:00-anchored trading date. A bar at/after 18:00 belongs to the next day."""
    return (dt - pd.Timedelta(hours=ANCHOR_HOUR)).dt.date


def build_ranges(bars: pd.DataFrame, duration: int, min_coverage: float = 0.60,
                 width_window: int = 250, atr_window: int = 14,
                 atr_rank_window: int = 250) -> pd.DataFrame:
    """
    Aggregate 1-minute bars into `duration`-minute ranges and attach features.

    bars         : Datetime (tz-aware ET), Open/High/Low/Close[/Volume]
    min_coverage : drop ranges holding fewer than this fraction of their
                   expected 1-min bars. Holiday half-days and the ragged ends of
                   a data file otherwise produce ranges built from a handful of
                   bars, which then dominate thin cells.
    width_window : trailing completed ranges used for width percentile rank.

    Returns one row per range, ordered by time.
    """
    if duration < 10:
        raise ValueError(f"duration must be >= 10 minutes, got {duration}")

    d = bars.loc[:, ["Datetime", "Open", "High", "Low", "Close"]].copy()
    d["Volume"] = bars["Volume"].to_numpy() if "Volume" in bars.columns else np.nan

    dt = d["Datetime"]
    mins = dt.dt.hour * 60 + dt.dt.minute
    since_anchor = (mins - ANCHOR_HOUR * 60) % 1440          # 0 at 18:00 ET
    sess = _session_date(dt)

    # A range is identified by its session date plus its slot index within the
    # session, so ranges never straddle the 18:00 boundary.
    slot_idx = (since_anchor // duration).astype(int)
    d["_sess"] = sess
    d["_slot_idx"] = slot_idx
    d["_rid"] = pd.factorize(
        pd.Series(sess).astype(str) + "_" + slot_idx.astype(str), sort=False)[0]

    g = d.groupby("_rid", sort=True)
    r = g.agg(
        start=("Datetime", "first"),
        end=("Datetime", "last"),
        Open=("Open", "first"),
        High=("High", "max"),
        Low=("Low", "min"),
        Close=("Close", "last"),
        Volume=("Volume", "sum"),
        n_bars=("Close", "size"),
    )
    r["sess"] = g["_sess"].first().to_numpy()
    r["slot_idx"] = g["_slot_idx"].first().to_numpy()

    # Which extreme formed first: compare the bar index of the high and the low.
    r["high_at"] = g["High"].idxmax().to_numpy()
    r["low_at"] = g["Low"].idxmin().to_numpy()
    r["formed_first"] = np.where(r["high_at"] < r["low_at"], "High", "Low")

    # Volume split either side of the range's own midpoint. Needs the mid, which
    # is only known once the range closes, so map it back onto the bars.
    r["mid"] = (r["High"] + r["Low"]) / 2.0
    mid_per_bar = d["_rid"].map(r["mid"])
    upper = d["Volume"].where(d["Close"] > mid_per_bar, 0.0)
    lower = d["Volume"].where(d["Close"] <= mid_per_bar, 0.0)
    r["vol_upper"] = upper.groupby(d["_rid"]).sum().to_numpy()
    r["vol_lower"] = lower.groupby(d["_rid"]).sum().to_numpy()

    r = r.reset_index(drop=True)

    # --- coverage filter -----------------------------------------------------
    r["coverage"] = r["n_bars"] / float(duration)
    r["ok_coverage"] = r["coverage"] >= min_coverage

    # --- geometry ------------------------------------------------------------
    r["width"] = r["High"] - r["Low"]
    r["width_pct_px"] = 100.0 * r["width"] / r["Open"].replace(0, np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        r["close_pos"] = (r["Close"] - r["Low"]) / r["width"].replace(0, np.nan)
    r["vol_balance"] = (r["vol_upper"] - r["vol_lower"]) / \
                       (r["vol_upper"] + r["vol_lower"]).replace(0, np.nan)

    # --- clock / calendar ----------------------------------------------------
    # `slot` must be the NOMINAL window boundary, not the timestamp of the first
    # bar that happens to be present. A window starting 23:45 whose first bars
    # are missing would otherwise be labelled 23:53 and become its own spurious
    # cell with n=1, polluting the grid and inflating the multiple-comparison
    # count. Derive it from the slot index instead.
    nominal = (ANCHOR_HOUR * 60 + r["slot_idx"] * duration) % 1440
    r["slot"] = (nominal // 60).astype(int).map("{:02d}".format) + ":" + \
                (nominal % 60).astype(int).map("{:02d}".format)
    r["dow"] = r["start"].dt.day_name()
    r["duration"] = duration

    # --- relation to the PREVIOUS range -------------------------------------
    # Only meaningful when the previous range is directly adjacent in time, the
    # same requirement the Range Probability indicator enforces via `sAdj`.
    pH, pL, pO, pC = (r[c].shift(1) for c in ("High", "Low", "Open", "Close"))
    p_end = r["end"].shift(1)
    gap_ok = (r["start"] - p_end).dt.total_seconds().between(0, 60 * duration)
    r["adjacent"] = gap_ok & r["ok_coverage"] & r["ok_coverage"].shift(1).fillna(False)

    pw = (pH - pL).replace(0, np.nan)
    r["op"] = (r["Open"] - pL) / pw                       # open position, 0=pLow 1=pHigh
    r["op_bucket"] = pd.cut(r["op"], bins=OP_EDGES, labels=OP_LABELS, right=False)
    r["prev_high"], r["prev_low"] = pH, pL
    r["prev_open"], r["prev_close"] = pO, pC
    r["prev_width"] = pw

    # Overlap / engulf relation. Cheap, and not in any of the existing notebooks.
    inside = (r["High"] <= pH) & (r["Low"] >= pL)
    engulf = (r["High"] >= pH) & (r["Low"] <= pL)
    above = r["Low"] > pH
    below = r["High"] < pL
    r["overlap_rel"] = np.select(
        [below, above, inside, engulf], ["gap below", "gap above", "inside prev", "engulfs prev"],
        default="overlaps")
    inter = (np.minimum(r["High"], pH) - np.maximum(r["Low"], pL)).clip(lower=0)
    r["overlap_frac"] = inter / pw

    # --- trailing normalisations (no lookahead: shift(1) then rolling) -------
    w = r["width"].where(r["ok_coverage"])
    trail = w.shift(1).rolling(width_window, min_periods=50)
    r["width_pct_rank"] = _trailing_pct_rank(w, width_window, min_periods=50)
    r["width_z"] = (w - trail.mean()) / trail.std(ddof=0)
    r["width_bucket"] = pd.cut(r["width_pct_rank"], [0, .2, .4, .6, .8, 1.0],
                               labels=["1 tightest", "2", "3", "4", "5 widest"],
                               include_lowest=True)

    rv = r["Volume"].where(r["ok_coverage"])
    r["rel_volume"] = rv / rv.shift(1).rolling(width_window, min_periods=50).median()

    # ATR on the range series itself, Wilder-smoothed, then percentile-ranked.
    prev_close = r["Close"].shift(1)
    tr = pd.concat([r["High"] - r["Low"],
                    (r["High"] - prev_close).abs(),
                    (r["Low"] - prev_close).abs()], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1.0 / atr_window, adjust=False, min_periods=atr_window).mean()
    r["atr"] = atr
    r["atr_pct_rank"] = _trailing_pct_rank(atr, atr_rank_window, min_periods=50)
    r["atr_zone"] = pd.cut(r["atr_pct_rank"], [0, .25, .5, .75, 1.0],
                           labels=["1 low", "2 mid-low", "3 mid-high", "4 high"],
                           include_lowest=True)

    return r


def _trailing_pct_rank(s: pd.Series, window: int, min_periods: int) -> pd.Series:
    """
    Fraction of the TRAILING `window` values strictly below the current value.

    Uses only prior observations, so it is safe to condition on. Equivalent in
    spirit to Pine's ta.percentrank but expressed as a 0-1 fraction.
    """
    v = s.to_numpy(dtype="float64")
    n = len(v)
    out = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - window)
        past = v[lo:i]
        past = past[~np.isnan(past)]
        if len(past) >= min_periods and not np.isnan(v[i]):
            out[i] = (past < v[i]).mean()
    return pd.Series(out, index=s.index)


def build_all(bars: pd.DataFrame, durations=(15, 30, 60, 120, 240), **kw) -> pd.DataFrame:
    """Build ranges for several durations and stack them."""
    frames = [build_ranges(bars, d, **kw) for d in durations]
    out = pd.concat(frames, ignore_index=True)
    out.attrs["symbol"] = bars.attrs.get("symbol", "UNKNOWN")
    return out

# ---- outcomes.py -------------------------------------------------
# Reversion targets tied to the range's own geometry. Their DISTANCE varies with
# the range width, which is why a tight range appears to "revert" more often.
TARGETS = ("mid", "range_open", "far_edge")

# Reversion targets at a fixed distance in ATRs back inside from the broken edge.
# These are directly comparable across slots and durations, because the distance
# is the same for every range. This is the honest version of the question
# "does price come back after a break".
ATR_TARGETS = (0.5, 1.0, 1.5)


def forward_outcomes(bars: pd.DataFrame, ranges: pd.DataFrame,
                     horizon_mult: float = 3.0, session_end=None
                     ) -> pd.DataFrame:
    """
    Attach forward outcomes to `ranges` (output of ranges.build_ranges).

    horizon_mult : forward window length, as a multiple of the range duration.
    session_end  : optional "HH:MM" ET cap, applied on top of the horizon. None
                   lets the horizon run across the 17:00-18:00 break.

    Returns `ranges` with the outcome columns appended.
    """
    # Work in tz-naive UTC datetime64 internally: a tz-aware Series converts to
    # object dtype under .to_numpy(), which breaks timedelta arithmetic.
    b_dt = _naive(bars["Datetime"])
    b_hi = bars["High"].to_numpy(dtype="float64")
    b_lo = bars["Low"].to_numpy(dtype="float64")
    n_bars = len(b_dt)

    r = ranges.reset_index(drop=True)
    duration = int(r["duration"].iloc[0])
    horizon = pd.Timedelta(minutes=duration * horizon_mult)

    r_end = _naive(r["end"])
    # forward window starts at the first bar strictly after the range's last bar
    start_i = np.searchsorted(b_dt, r_end, side="right")
    stop_ts = r_end + horizon.to_timedelta64()
    if session_end is not None:
        cap = _naive(_session_end_ts(r["end"], session_end))
        stop_ts = np.minimum(stop_ts, cap)
    stop_i = np.searchsorted(b_dt, stop_ts, side="right")

    hi = r["High"].to_numpy(dtype="float64")
    lo = r["Low"].to_numpy(dtype="float64")
    op = r["Open"].to_numpy(dtype="float64")
    atr = r["atr"].to_numpy(dtype="float64") if "atr" in r.columns else np.full(len(r), np.nan)
    mid = ((hi + lo) / 2.0)
    width = (hi - lo)

    n = len(r)
    out = {
        "break_side": np.full(n, "", dtype=object),
        "break_ambiguous": np.zeros(n, dtype=bool),
        "mins_to_break": np.full(n, np.nan),
        "cont_R": np.full(n, np.nan),          # continuation beyond the edge, in R
        "mae_to_mid_R": np.full(n, np.nan),    # walk-away adverse excursion, in R
        "terminal_R": np.full(n, np.nan),      # where price ended vs the broken edge
        "fwd_bars": np.zeros(n, dtype=int),
    }
    for t in TARGETS:
        out[f"rev_{t}"] = np.full(n, np.nan)        # 1/0, NaN if no break
        out[f"mins_to_{t}"] = np.full(n, np.nan)
    for a in ATR_TARGETS:
        tag = _atr_tag(a)
        out[f"rev_{tag}"] = np.full(n, np.nan)
        out[f"mins_to_{tag}"] = np.full(n, np.nan)
    out["cont_atr"] = np.full(n, np.nan)            # travel past the edge, in ATR

    for k in range(n):
        s, e = int(start_i[k]), int(stop_i[k])
        if s >= n_bars or e <= s or not np.isfinite(width[k]) or width[k] <= 0:
            continue
        out["fwd_bars"][k] = e - s
        H, L = b_hi[s:e], b_lo[s:e]

        up = H > hi[k]
        dn = L < lo[k]
        i_up = int(np.argmax(up)) if up.any() else -1
        i_dn = int(np.argmax(dn)) if dn.any() else -1

        if i_up < 0 and i_dn < 0:
            out["break_side"][k] = "none"
            continue

        if i_up >= 0 and i_dn >= 0 and i_up == i_dn:
            # same bar pierced both edges: take the nearer edge to that bar's
            # own range, flag it, and carry on.
            out["break_ambiguous"][k] = True
            bar_mid = (H[i_up] + L[i_up]) / 2.0
            side = "High" if abs(bar_mid - hi[k]) <= abs(bar_mid - lo[k]) else "Low"
            bi = i_up
        elif i_dn < 0 or (i_up >= 0 and i_up < i_dn):
            side, bi = "High", i_up
        else:
            side, bi = "Low", i_dn

        out["break_side"][k] = side
        out["mins_to_break"][k] = (b_dt[s + bi] - r_end[k]) / np.timedelta64(1, "m")

        post_H, post_L = H[bi:], L[bi:]
        if side == "High":
            edge, far = hi[k], lo[k]
            out["cont_R"][k] = (post_H.max() - edge) / width[k]
            out["terminal_R"][k] = (b_hi[e - 1] + b_lo[e - 1]) / 2.0
            out["terminal_R"][k] = (out["terminal_R"][k] - edge) / width[k]
            tgt_vals = {"mid": mid[k], "range_open": op[k], "far_edge": far}
            if np.isfinite(atr[k]) and atr[k] > 0:
                for a in ATR_TARGETS:
                    tgt_vals[_atr_tag(a)] = edge - a * atr[k]
                out["cont_atr"][k] = (post_H.max() - edge) / atr[k]
            # reversion is a move DOWN to the target, strictly after the break bar
            for t, lvl in tgt_vals.items():
                hit = np.flatnonzero(post_L[1:] <= lvl)
                _record(out, k, t, hit, bi, s, b_dt, r_end)
            # walk-away: adverse (further up) only until the mid is reached
            stop_at = _first_idx(post_L[1:] <= mid[k])
            seg = post_H[:stop_at + 2] if stop_at is not None else post_H
            out["mae_to_mid_R"][k] = (seg.max() - edge) / width[k]
        else:
            edge, far = lo[k], hi[k]
            out["cont_R"][k] = (edge - post_L.min()) / width[k]
            tm = (b_hi[e - 1] + b_lo[e - 1]) / 2.0
            out["terminal_R"][k] = (edge - tm) / width[k]
            tgt_vals = {"mid": mid[k], "range_open": op[k], "far_edge": far}
            if np.isfinite(atr[k]) and atr[k] > 0:
                for a in ATR_TARGETS:
                    tgt_vals[_atr_tag(a)] = edge + a * atr[k]
                out["cont_atr"][k] = (edge - post_L.min()) / atr[k]
            for t, lvl in tgt_vals.items():
                hit = np.flatnonzero(post_H[1:] >= lvl)
                _record(out, k, t, hit, bi, s, b_dt, r_end)
            stop_at = _first_idx(post_H[1:] >= mid[k])
            seg = post_L[:stop_at + 2] if stop_at is not None else post_L
            out["mae_to_mid_R"][k] = (edge - seg.min()) / width[k]

    res = r.copy()
    for col, arr in out.items():
        res[col] = arr
    res["broke"] = res["break_side"].isin(["High", "Low"])
    res["horizon_mult"] = horizon_mult

    # Forward coverage. Slots late in the session have their horizon cut short by
    # the 17:00 close and the maintenance break, so they get fewer chances to
    # break or revert. Without this, "did not break" at 16:45 looks like a
    # signal when it is really an absence of data.
    expected_fwd = duration * horizon_mult
    res["fwd_coverage"] = res["fwd_bars"] / expected_fwd
    res["ok_forward"] = res["fwd_coverage"] >= 0.90
    res["usable"] = res["ok_coverage"] & res["ok_forward"] & res["adjacent"]

    # Reversion flags are only defined conditional on a break.
    for t in list(TARGETS) + [_atr_tag(a) for a in ATR_TARGETS]:
        res.loc[~res["broke"], f"rev_{t}"] = np.nan

    # Magnitude-conditioned outcomes. A 70% direction call that travels 0.2R is
    # not a trade idea, so these ask whether the break actually went somewhere.
    broke = res["broke"].to_numpy()
    for thr in (0.5, 1.0, 2.0):
        tag = f"cont_{str(thr).replace('.', 'p')}R"
        res[tag] = np.where(broke, (res["cont_R"].to_numpy() >= thr).astype("float64"), np.nan)
    for a in ATR_TARGETS:
        res[f"cont_{str(a).replace('.', 'p')}atr"] = np.where(
            broke, (res["cont_atr"].to_numpy() >= a).astype("float64"), np.nan)
    is_up = (res["break_side"] == "High").to_numpy()
    res["up_and_1R"] = np.where(broke, (is_up & (res["cont_R"].to_numpy() >= 1.0)).astype("float64"), np.nan)
    res["dn_and_1R"] = np.where(broke, (~is_up & (res["cont_R"].to_numpy() >= 1.0)).astype("float64"), np.nan)
    # Did it continue rather than revert? The cleanest breakout-vs-reversion split.
    res["cont_not_rev"] = np.where(
        broke, ((res["cont_R"].to_numpy() >= 1.0) & (res["rev_mid"].to_numpy() == 0)).astype("float64"), np.nan)
    return res


def _naive(s: pd.Series) -> np.ndarray:
    """tz-aware datetime Series -> tz-naive UTC datetime64[ns] array."""
    s = pd.to_datetime(s)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s.to_numpy("datetime64[ns]")


def _atr_tag(a):
    """0.5 -> 'back_0p5atr'"""
    return "back_" + str(a).replace(".", "p") + "atr"


def _first_idx(mask):
    nz = np.flatnonzero(mask)
    return int(nz[0]) if len(nz) else None


def _record(out, k, t, hit, bi, s, b_dt, r_end):
    if len(hit):
        out[f"rev_{t}"][k] = 1.0
        j = s + bi + 1 + int(hit[0])
        out[f"mins_to_{t}"][k] = (b_dt[j] - r_end[k]) / np.timedelta64(1, "m")
    else:
        out[f"rev_{t}"][k] = 0.0


def _session_end_ts(end: pd.Series, session_end: str) -> pd.Series:
    """Timestamp of `session_end` on the trading day each range belongs to."""
    hh, mm = (int(x) for x in session_end.split(":"))
    anchored = end - pd.Timedelta(hours=18)
    day = anchored.dt.normalize()
    same = day + pd.Timedelta(hours=18) + pd.Timedelta(hours=hh, minutes=mm)
    nxt = same + pd.Timedelta(days=1)
    return pd.Series(np.where(same > end, same, nxt), index=end.index)



# =====================================================================
# hit_rates - group ranges any way you like, get hit rates back
# =====================================================================
def hit_rates(B, by=("duration", "slot"), min_n=30, sort="n"):
    """
    Descriptive hit rates for ranges that broke an edge.

    Every continuation / reversion figure comes three ways:
      <name>        both directions pooled
      <name>_up     only the ranges that broke the HIGH
      <name>_dn     only the ranges that broke the LOW

    so you can ask "when this window breaks DOWN, does it keep going?"
    separately from the upside.

    by   : how to group. Anything in B works, e.g.
           ("duration","slot")              per window
           ("duration","slot","dow")        per window per weekday
           ("duration","close_pos_b")       by where the range closed
    min_n: drop groups with fewer than this many ranges
    """
    d = B.copy()
    up = d["up"].to_numpy() == 1

    # direction-split copies: NaN on the other side, so .mean() skips them
    split_cols = ["cont_0p5atr", "cont_1p0atr", "cont_1p5atr",
                  "rev_back_0p5atr", "rev_back_1p0atr", "rev_back_1p5atr",
                  "rev_mid", "rev_far_edge", "cont_atr"]
    for c in split_cols:
        v = d[c].to_numpy(dtype="float64")
        d[c + "__up"] = np.where(up, v, np.nan)
        d[c + "__dn"] = np.where(~up, v, np.nan)

    g = d.groupby(list(by), observed=True)
    t = g.agg(
        n                = ("up", "size"),
        n_up             = ("up", "sum"),
        break_high_pct   = ("up", "mean"),
        mins_to_break    = ("mins_to_break", "median"),
        travel_atr_med   = ("cont_atr", "median"),
        # ---- pooled ----
        cont_0_5atr      = ("cont_0p5atr", "mean"),
        cont_1atr        = ("cont_1p0atr", "mean"),
        cont_1_5atr      = ("cont_1p5atr", "mean"),
        back_0_5atr      = ("rev_back_0p5atr", "mean"),
        back_1atr        = ("rev_back_1p0atr", "mean"),
        back_1_5atr      = ("rev_back_1p5atr", "mean"),
        back_to_mid      = ("rev_mid", "mean"),
        back_to_far_edge = ("rev_far_edge", "mean"),
        mins_to_mid      = ("mins_to_mid", "median"),
        # ---- UP breaks only ----
        cont_0_5atr_up   = ("cont_0p5atr__up", "mean"),
        cont_1atr_up     = ("cont_1p0atr__up", "mean"),
        cont_1_5atr_up   = ("cont_1p5atr__up", "mean"),
        back_0_5atr_up   = ("rev_back_0p5atr__up", "mean"),
        back_1atr_up     = ("rev_back_1p0atr__up", "mean"),
        back_to_mid_up   = ("rev_mid__up", "mean"),
        travel_atr_up    = ("cont_atr__up", "median"),
        # ---- DOWN breaks only ----
        cont_0_5atr_dn   = ("cont_0p5atr__dn", "mean"),
        cont_1atr_dn     = ("cont_1p0atr__dn", "mean"),
        cont_1_5atr_dn   = ("cont_1p5atr__dn", "mean"),
        back_0_5atr_dn   = ("rev_back_0p5atr__dn", "mean"),
        back_1atr_dn     = ("rev_back_1p0atr__dn", "mean"),
        back_to_mid_dn   = ("rev_mid__dn", "mean"),
        travel_atr_dn    = ("cont_atr__dn", "median"),
        width_med        = ("width", "median"),
        width_atr_med    = ("w_atr", "median"),
    ).reset_index()

    t["n_dn"] = t["n"] - t["n_up"]
    t["break_low_pct"] = 1 - t["break_high_pct"]
    # lean measured against the OVERALL base rate, not 50. NQ breaks high
    # 52.2% of the time, so a 52% window is average, not bullish.
    base_up = float(B["up"].mean())
    t["base_high_pct"] = base_up
    t["lean_pp"] = t["break_high_pct"] - base_up        # + = leans UP, - = leans DOWN

    t = t[t.n >= min_n]
    for c in t.columns:
        if c.endswith(("_pct", "atr", "_mid", "_edge", "_up", "_dn", "_pp")) \
           and not c.startswith(("travel", "width", "mins", "n_")):
            t[c] = (100 * t[c]).round(1)
    for c in ("mins_to_break", "mins_to_mid"):
        t[c] = t[c].round(0)
    for c in ("travel_atr_med", "travel_atr_up", "travel_atr_dn", "width_atr_med"):
        t[c] = t[c].round(2)
    t["width_med"] = t["width_med"].round(1)

    if sort in t.columns:
        t = t.sort_values(sort, ascending=(sort == "n"))
    return t.reset_index(drop=True)


KEY  = ["duration", "slot", "n"]
UP   = ["n_up", "cont_1atr_up", "back_1atr_up", "travel_atr_up", "back_to_mid_up"]
DN   = ["n_dn", "cont_1atr_dn", "back_1atr_dn", "travel_atr_dn", "back_to_mid_dn"]
BOTH = ["break_high_pct", "lean_pp", "mins_to_break"]

## Cell 2 — build the ranges

In [2]:
NQ         = "nq_15y_1min.csv"
DURATIONS  = (15, 30, 60, 120, 240)
START, END = "2013-01-01", "2025-03-14"
HORIZON    = 3.0        # forward window = HORIZON x the range duration

bars = load_bars(NQ, symbol="NQ", start=START, end=END)
print(f"{len(bars):,} bars  {bars['Datetime'].iloc[0]:%Y-%m-%d} -> {bars['Datetime'].iloc[-1]:%Y-%m-%d}")

frames = []
for dur in DURATIONS:
    frames.append(forward_outcomes(bars, build_ranges(bars, dur), horizon_mult=HORIZON))
    print(f"  {dur:>4}-min done")

R = pd.concat(frames, ignore_index=True)
R = R[R["usable"]].copy()
B = R[R["broke"]].copy()
B["up"]    = (B["break_side"] == "High").astype(float)
B["w_atr"] = B["width"] / B["atr"]

for col, q in [("close_pos", 4), ("vol_balance", 4), ("rel_volume", 4), ("w_atr", 5)]:
    B[col + "_b"] = pd.qcut(B[col], q, labels=[f"{i+1}" for i in range(q)],
                            duplicates="drop").astype(str)

n_up, n_dn = int(B["up"].sum()), int((1-B["up"]).sum())
print(f"\n{len(R):,} usable ranges | {len(B):,} broke an edge")
print(f"  broke the HIGH: {n_up:,} ({100*n_up/len(B):.1f}%)")
print(f"  broke the LOW : {n_dn:,} ({100*n_dn/len(B):.1f}%)")

4,160,708 bars  2013-01-02 -> 2025-03-14
    15-min done
    30-min done
    60-min done
   120-min done
   240-min done

443,513 usable ranges | 431,365 broke an edge
  broke the HIGH: 225,233 (52.2%)
  broke the LOW : 206,132 (47.8%)


## Cell 3 — overall, both directions

If the up and down columns look similar, the market is treating both sides alike
and direction is not where the information is.

In [3]:
up = B[B.up == 1]; dn = B[B.up == 0]
print(f"{'':22s}{'UP breaks':>12s}{'DOWN breaks':>14s}")
print(f"{'count':22s}{len(up):>12,}{len(dn):>14,}")
print(f"{'median mins to break':22s}{up.mins_to_break.median():>12.0f}{dn.mins_to_break.median():>14.0f}")
print(f"{'median travel (ATR)':22s}{up.cont_atr.median():>12.2f}{dn.cont_atr.median():>14.2f}")
print()
for a, tag in ((0.5,"0p5"), (1.0,"1p0"), (1.5,"1p5")):
    print(f"continues {a:>3.1f} ATR     {100*up[f'cont_{tag}atr'].mean():>11.1f}%{100*dn[f'cont_{tag}atr'].mean():>13.1f}%")
for a, tag in ((0.5,"0p5"), (1.0,"1p0"), (1.5,"1p5")):
    print(f"comes back {a:>3.1f} ATR    {100*up[f'rev_back_{tag}atr'].mean():>11.1f}%{100*dn[f'rev_back_{tag}atr'].mean():>13.1f}%")
print(f"{'back to range mid':22s}{100*up.rev_mid.mean():>11.1f}%{100*dn.rev_mid.mean():>13.1f}%")
print(f"{'back to far edge':22s}{100*up.rev_far_edge.mean():>11.1f}%{100*dn.rev_far_edge.mean():>13.1f}%")

                         UP breaks   DOWN breaks
count                      225,233       206,132
median mins to break             5             5
median travel (ATR)           0.66          0.69

continues 0.5 ATR            60.4%         61.3%
continues 1.0 ATR            33.0%         36.0%
continues 1.5 ATR            18.0%         21.9%
comes back 0.5 ATR           54.3%         58.8%
comes back 1.0 ATR           29.5%         32.8%
comes back 1.5 ATR           16.6%         18.0%
back to range mid            59.2%         63.4%
back to far edge             34.7%         37.5%


## Cell 4 — the main table

One row per (duration, slot), with `_up` and `_dn` columns alongside the pooled ones.

In [4]:
T = hit_rates(B, by=("duration","slot"), min_n=50)
print(f"{len(T)} windows | base rate for breaking HIGH: {T.base_high_pct.iloc[0]}%\n")
T

162 windows | base rate for breaking HIGH: 52.2%



,duration,slot,n,n_up,break_high_pct,mins_to_break,travel_atr_med,cont_0_5atr,cont_1atr,cont_1_5atr,...,back_0_5atr_dn,back_1atr_dn,back_to_mid_dn,travel_atr_dn,width_med,width_atr_med,n_dn,break_low_pct,base_high_pct,lean_pp
0,240,10:00,605,341.0,56.4,33.0,0.49,49.1,23.6,10.2,...,49.6,21.6,31.8,0.55,155.5,1.48,264.0,43.6,52.2,4.1
1,15,16:00,616,329.0,53.4,8.0,0.19,17.2,6.7,4.1,...,15.0,4.5,25.8,0.21,23.6,0.68,287.0,46.6,52.2,1.2
2,15,15:45,662,349.0,52.7,1.0,0.37,37.6,14.8,7.1,...,29.1,10.2,21.1,0.38,44.0,1.21,313.0,47.3,52.2,0.5
3,240,06:00,760,373.0,49.1,9.0,0.89,70.1,45.3,25.1,...,67.2,43.7,57.9,0.92,125.2,1.24,387.0,50.9,52.2,-3.1
4,30,15:00,891,446.0,50.1,6.0,0.57,54.9,26.2,13.1,...,57.1,27.4,56.6,0.57,44.0,0.98,445.0,49.9,52.2,-2.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,30,08:00,3114,1591.0,51.1,3.0,1.77,85.8,70.4,56.4,...,80.1,66.1,79.5,1.79,12.2,1.04,1523.0,48.9,52.2,-1.1
158,15,08:45,3115,1583.0,50.8,4.0,1.73,85.7,70.4,56.2,...,84.2,70.1,82.5,1.69,9.2,1.09,1532.0,49.2,52.2,-1.4
159,30,09:00,3119,1560.0,50.0,1.0,2.08,88.4,75.5,63.2,...,85.2,73.4,83.3,2.10,13.2,1.12,1559.0,50.0,52.2,-2.2
160,15,09:00,3120,1574.0,50.4,5.0,2.08,87.6,74.9,62.5,...,86.9,74.2,85.3,2.06,9.2,1.08,1546.0,49.6,52.2,-1.8


### Windows that lean DOWN the most

`lean_pp` is relative to the base rate, so these genuinely favour the downside.

In [5]:
T.sort_values("lean_pp").head(20)[
    KEY + ["break_high_pct","break_low_pct","lean_pp","n_dn",
           "cont_1atr_dn","back_1atr_dn","travel_atr_dn"]]

,duration,slot,n,break_high_pct,break_low_pct,lean_pp,n_dn,cont_1atr_dn,back_1atr_dn,travel_atr_dn
135,15,02:45,3041,45.3,54.7,-6.9,1662.0,58.5,53.9,1.25
146,30,02:30,3066,47.2,52.8,-5.0,1619.0,44.6,46.5,0.89
6,15,15:30,912,48.2,51.8,-4.0,472.0,29.7,31.4,0.61
69,15,08:30,2891,48.8,51.2,-3.4,1479.0,41.5,39.4,0.79
85,60,02:00,2965,48.8,51.2,-3.4,1519.0,31.3,33.6,0.65
138,15,07:30,3047,48.8,51.2,-3.4,1559.0,39.6,34.9,0.81
3,240,06:00,760,49.1,50.9,-3.1,387.0,46.5,43.7,0.92
121,15,03:45,3016,49.2,50.8,-3.0,1531.0,45.8,37.8,0.90
149,30,07:30,3078,49.2,50.8,-3.0,1564.0,45.3,43.6,0.88
151,15,07:45,3083,49.4,50.6,-2.8,1559.0,47.7,45.9,0.94


### Windows that lean UP the most

In [6]:
T.sort_values("lean_pp", ascending=False).head(20)[
    KEY + ["break_high_pct","break_low_pct","lean_pp","n_up",
           "cont_1atr_up","back_1atr_up","travel_atr_up"]]

,duration,slot,n,break_high_pct,break_low_pct,lean_pp,n_up,cont_1atr_up,back_1atr_up,travel_atr_up
0,240,10:00,605,56.4,43.6,4.1,341.0,18.5,13.8,0.47
10,240,18:00,2122,56.2,43.8,4.0,1193.0,25.4,26.8,0.60
97,60,12:00,2984,55.5,44.5,3.2,1655.0,40.1,37.4,0.80
73,60,11:00,2911,55.2,44.8,3.0,1606.0,36.7,35.2,0.75
141,30,06:30,3055,55.2,44.8,3.0,1687.0,37.9,30.4,0.74
143,15,11:00,3060,55.2,44.8,2.9,1688.0,33.3,31.5,0.72
7,240,14:00,1316,55.1,44.9,2.9,725.0,9.1,8.7,0.40
152,120,04:00,3097,55.1,44.9,2.8,1705.0,53.7,54.5,1.10
104,30,14:00,2986,55.1,44.9,2.8,1644.0,35.7,35.4,0.77
137,15,06:45,3044,55.1,44.9,2.8,1676.0,37.3,28.8,0.77


### Best DOWNSIDE breakouts — breaks low and keeps going

In [7]:
T[T.n_dn >= 50].sort_values("cont_1atr_dn", ascending=False).head(20)[
    KEY + ["n_dn","break_low_pct","cont_1atr_dn","cont_1_5atr_dn","travel_atr_dn","back_1atr_dn"]]

,duration,slot,n,n_dn,break_low_pct,cont_1atr_dn,cont_1_5atr_dn,travel_atr_dn,back_1atr_dn
161,15,09:15,3121,1574.0,50.4,78.0,67.2,2.42,74.5
160,15,09:00,3120,1546.0,49.6,74.3,61.1,2.06,74.2
159,30,09:00,3119,1559.0,50.0,74.1,63.8,2.10,73.4
153,30,08:30,3098,1508.0,48.7,71.9,60.1,1.90,71.8
158,15,08:45,3115,1532.0,49.2,70.0,54.7,1.69,70.1
157,30,08:00,3114,1523.0,48.9,69.6,56.9,1.79,66.1
119,15,09:30,3014,1460.0,48.4,68.2,52.7,1.59,63.2
156,60,07:00,3111,1513.0,48.6,68.0,52.9,1.61,64.2
155,60,08:00,3105,1494.0,48.1,67.2,51.9,1.57,66.7
145,15,09:45,3061,1427.0,46.6,64.8,47.9,1.42,59.1


### Best UPSIDE breakouts — breaks high and keeps going

In [8]:
T[T.n_up >= 50].sort_values("cont_1atr_up", ascending=False).head(20)[
    KEY + ["n_up","break_high_pct","cont_1atr_up","cont_1_5atr_up","travel_atr_up","back_1atr_up"]]

,duration,slot,n,n_up,break_high_pct,cont_1atr_up,cont_1_5atr_up,travel_atr_up,back_1atr_up
161,15,09:15,3121,1547.0,49.6,80.7,68.7,2.30,71.4
159,30,09:00,3119,1560.0,50.0,76.9,62.6,2.05,70.5
160,15,09:00,3120,1574.0,50.4,75.5,63.9,2.09,72.2
157,30,08:00,3114,1591.0,51.1,71.2,55.9,1.73,66.6
153,30,08:30,3098,1590.0,51.3,71.0,56.0,1.72,70.5
158,15,08:45,3115,1583.0,50.8,70.8,57.7,1.76,68.5
156,60,07:00,3111,1598.0,51.4,67.0,50.0,1.50,64.2
155,60,08:00,3105,1611.0,51.9,64.6,49.8,1.49,63.9
119,15,09:30,3014,1554.0,51.6,64.2,46.1,1.39,57.9
154,60,06:00,3099,1687.0,54.4,59.4,40.6,1.24,55.2


### Best DOWNSIDE reversions — breaks low then comes back

In [9]:
T[T.n_dn >= 50].sort_values("back_1atr_dn", ascending=False).head(20)[
    KEY + ["n_dn","back_0_5atr_dn","back_1atr_dn","back_to_mid_dn","cont_1atr_dn"]]

,duration,slot,n,n_dn,back_0_5atr_dn,back_1atr_dn,back_to_mid_dn,cont_1atr_dn
161,15,09:15,3121,1574.0,84.9,74.5,82.8,78.0
160,15,09:00,3120,1546.0,86.9,74.2,85.3,74.3
159,30,09:00,3119,1559.0,85.2,73.4,83.3,74.1
153,30,08:30,3098,1508.0,86.3,71.8,79.6,71.9
158,15,08:45,3115,1532.0,84.2,70.1,82.5,70.0
155,60,08:00,3105,1494.0,81.9,66.7,79.7,67.2
157,30,08:00,3114,1523.0,80.1,66.1,79.5,69.6
156,60,07:00,3111,1513.0,80.5,64.2,82.7,68.0
119,15,09:30,3014,1460.0,77.7,63.2,48.4,68.2
87,30,09:30,2969,1378.0,78.6,61.0,48.2,64.3


### Best UPSIDE reversions — breaks high then comes back

In [10]:
T[T.n_up >= 50].sort_values("back_1atr_up", ascending=False).head(20)[
    KEY + ["n_up","back_0_5atr_up","back_1atr_up","back_to_mid_up","cont_1atr_up"]]

,duration,slot,n,n_up,back_0_5atr_up,back_1atr_up,back_to_mid_up,cont_1atr_up
160,15,09:00,3120,1574.0,85.2,72.2,84.2,75.5
161,15,09:15,3121,1547.0,83.0,71.4,80.9,80.7
153,30,08:30,3098,1590.0,85.7,70.5,79.2,71.0
159,30,09:00,3119,1560.0,82.5,70.5,80.3,76.9
158,15,08:45,3115,1583.0,84.9,68.5,81.7,70.8
157,30,08:00,3114,1591.0,80.6,66.6,79.5,71.2
156,60,07:00,3111,1598.0,80.2,64.2,82.8,67.0
155,60,08:00,3105,1611.0,80.6,63.9,77.6,64.6
120,120,06:00,3014,1601.0,76.6,59.0,80.9,58.6
119,15,09:30,3014,1554.0,75.8,57.9,45.1,64.2


### Where the two sides behave most differently

Windows where an up-break and a down-break lead to different outcomes. These are
the ones worth a second look, because a window that behaves the same both ways is
telling you about volatility, not direction.

In [11]:
X = T[(T.n_up >= 50) & (T.n_dn >= 50)].copy()
X["cont_gap"] = X.cont_1atr_up - X.cont_1atr_dn     # + = upside follows through more
X["back_gap"] = X.back_1atr_up - X.back_1atr_dn
X.reindex(X.cont_gap.abs().sort_values(ascending=False).index).head(20)[
    KEY + ["break_high_pct","cont_1atr_up","cont_1atr_dn","cont_gap",
           "back_1atr_up","back_1atr_dn","back_gap"]]

,duration,slot,n,break_high_pct,cont_1atr_up,cont_1atr_dn,cont_gap,back_1atr_up,back_1atr_dn,back_gap
0,240,10:00,605,56.4,18.5,30.3,-11.8,13.8,21.6,-7.8
82,30,12:00,2963,54.7,28.7,38.0,-9.3,27.9,36.7,-8.8
73,60,11:00,2911,55.2,36.7,45.8,-9.1,35.2,45.2,-10.0
110,15,13:15,2996,53.6,27.9,36.2,-8.3,26.5,28.7,-2.2
10,240,18:00,2122,56.2,25.4,33.2,-7.8,26.8,30.4,-3.6
106,15,12:45,2992,52.7,24.4,31.7,-7.3,24.2,29.8,-5.6
115,15,12:15,3001,54.4,23.8,31.1,-7.3,23.0,27.7,-4.7
98,30,12:30,2985,53.1,33.9,41.1,-7.2,31.0,38.0,-7.0
93,15,13:30,2978,51.8,28.2,35.4,-7.2,27.8,29.5,-1.7
66,60,10:00,2813,54.4,41.1,48.2,-7.1,36.3,47.9,-11.6


## Cell 5 — slice by anything else

Any column in `B` works. Ready-made: `slot`, `dow`, `w_atr_b` (width vs ATR,
1 = tightest), `close_pos_b` (where it closed inside itself, 1 = at its low),
`vol_balance_b`, `rel_volume_b`, `formed_first`, `overlap_rel`, `atr_zone`.

In [12]:
# where the range closed inside itself - the strongest directional conditioner here
hit_rates(B, by=("duration","close_pos_b"), min_n=50)[
    ["duration","close_pos_b","n","break_high_pct","lean_pp",
     "cont_1atr_up","cont_1atr_dn","back_1atr_up","back_1atr_dn"]]

,duration,close_pos_b,n,break_high_pct,lean_pp,cont_1atr_up,cont_1atr_dn,back_1atr_up,back_1atr_dn
0,240,1,2160,15.1,-37.1,32.7,39.5,36.1,38.3
1,240,2,2215,40.6,-11.6,34.1,39.8,35.3,36.7
2,240,3,2393,67.1,14.9,35.8,37.4,34.0,36.8
3,240,4,2521,89.8,37.6,33.3,39.3,33.8,33.5
4,120,1,4624,15.5,-36.8,38.5,43.8,37.3,43.9
5,120,2,4992,39.9,-12.3,37.5,42.1,37.7,40.0
6,120,4,5185,87.5,35.3,40.0,42.1,39.4,35.2
7,120,3,5188,67.3,15.1,37.6,39.5,36.4,38.3
8,60,1,12515,14.6,-37.7,32.4,38.5,27.9,37.7
9,60,2,13135,40.1,-12.1,32.5,37.4,30.5,34.7


In [13]:
# which extreme formed first
hit_rates(B, by=("duration","formed_first"), min_n=50)[
    ["duration","formed_first","n","break_high_pct","lean_pp",
     "cont_1atr_up","cont_1atr_dn","back_1atr_up","back_1atr_dn"]]

,duration,formed_first,n,break_high_pct,lean_pp,cont_1atr_up,cont_1atr_dn,back_1atr_up,back_1atr_dn
0,240,High,4325,36.0,-16.3,33.8,40.0,35.1,36.8
1,240,Low,4964,71.3,19.1,34.4,37.7,33.9,37.9
2,120,High,9426,35.0,-17.2,38.3,42.9,38.0,42.2
3,120,Low,10563,70.4,18.2,38.8,41.2,38.0,38.8
4,60,High,25123,33.8,-18.4,33.4,38.6,30.8,36.4
5,60,Low,27353,70.7,18.5,34.7,35.5,31.3,33.7
6,30,High,54603,33.9,-18.3,31.5,37.0,27.6,33.8
7,30,Low,58060,69.6,17.4,33.3,33.1,29.7,30.7
8,15,High,115673,33.7,-18.5,30.7,36.2,27.7,32.4
9,15,Low,121275,69.0,16.8,33.0,33.3,28.8,29.2


In [14]:
# range width vs its own ATR
hit_rates(B, by=("duration","w_atr_b"), min_n=50)[
    ["duration","w_atr_b","n","width_atr_med","break_high_pct",
     "cont_1atr_up","cont_1atr_dn","back_1atr_up","back_1atr_dn"]]

,duration,w_atr_b,n,width_atr_med,break_high_pct,cont_1atr_up,cont_1atr_dn,back_1atr_up,back_1atr_dn
0,240,5,1182,1.65,52.4,22.8,36.2,22.5,32.3
1,240,4,1220,1.15,55.8,27.6,36.2,29.7,33.8
2,240,3,1346,0.87,55.8,33.3,40.8,30.8,36.3
3,240,2,1903,0.66,55.3,36.5,37.5,37.7,36.6
4,120,4,2789,1.13,53.7,44.6,49.5,46.2,48.3
5,120,5,3247,1.73,50.8,48.5,55.0,49.3,56.2
6,120,3,3281,0.87,54.1,43.2,45.6,43.4,44.9
7,240,1,3634,0.41,54.7,39.2,41.5,39.0,40.6
8,120,2,4582,0.66,54.4,38.6,42.9,39.1,39.6
9,120,1,6085,0.43,54.5,28.8,29.5,24.8,27.8


In [15]:
# how this range sat against the previous one
hit_rates(B, by=("duration","overlap_rel"), min_n=50)[
    ["duration","overlap_rel","n","break_high_pct","lean_pp",
     "cont_1atr_up","cont_1atr_dn","back_1atr_up","back_1atr_dn"]]

,duration,overlap_rel,n,break_high_pct,lean_pp,cont_1atr_up,cont_1atr_dn,back_1atr_up,back_1atr_dn
0,15,gap below,100,37.0,-15.2,16.2,38.1,16.2,23.8
1,15,gap above,127,54.3,2.1,29.0,13.8,20.3,29.3
2,240,engulfs prev,1065,51.3,-0.9,43.6,46.6,40.5,45.5
3,240,inside prev,2236,54.7,2.5,33.7,37.2,34.4,33.6
4,120,inside prev,3071,53.2,1.0,35.2,38.6,32.7,34.9
5,120,engulfs prev,3492,51.3,-1.0,41.5,46.9,42.6,46.5
6,240,overlaps,5969,55.6,3.3,32.9,38.6,33.2,37.0
7,60,engulfs prev,6744,51.9,-0.3,38.5,41.6,35.9,41.2
8,60,inside prev,8569,53.0,0.8,30.7,34.1,27.8,31.7
9,30,engulfs prev,13365,51.9,-0.3,36.8,39.8,32.3,37.0


In [16]:
# slot AND close position together
hit_rates(B, by=("duration","slot","close_pos_b"), min_n=100).sort_values("lean_pp").head(25)[
    ["duration","slot","close_pos_b","n","break_high_pct","lean_pp","cont_1atr_dn","back_1atr_dn"]]

,duration,slot,close_pos_b,n,break_high_pct,lean_pp,cont_1atr_dn,back_1atr_dn
14,15,15:45,1,200,5.5,-46.7,14.8,10.6
6,240,10:00,1,152,7.2,-45.0,34.0,18.4
30,240,14:00,1,335,9.3,-43.0,18.1,15.8
18,240,06:00,1,214,9.8,-42.4,47.7,45.1
489,30,09:30,1,759,10.4,-41.8,65.0,65.0
226,60,11:00,1,645,10.7,-41.5,44.1,48.5
538,60,09:00,1,774,11.1,-41.1,59.2,60.9
8,15,16:00,1,166,11.4,-40.8,7.5,4.8
296,30,12:00,1,695,11.4,-40.8,40.9,40.6
420,15,10:30,1,735,11.4,-40.8,49.3,46.2


## Cell 6 — save

In [17]:
hit_rates(B, by=("duration","slot"), min_n=50).to_csv("hit_rates_by_window.csv", index=False)
hit_rates(B, by=("duration","slot","dow"), min_n=30).to_csv("hit_rates_by_window_dow.csv", index=False)
hit_rates(B, by=("duration","slot","close_pos_b"), min_n=50).to_csv("hit_rates_by_window_closepos.csv", index=False)
print("written 3 csv files")

written 3 csv files


## Column reference

Suffix `_up` = only ranges that broke the HIGH. `_dn` = only those that broke the
LOW. No suffix = both pooled.

| column | meaning |
|---|---|
| `n`, `n_up`, `n_dn` | ranges in the group, and the split |
| `break_high_pct` / `break_low_pct` | which edge went first |
| `lean_pp` | `break_high_pct` minus the overall base. **+ leans up, − leans down** |
| `mins_to_break` | median minutes from range close to the break |
| `travel_atr_med` / `_up` / `_dn` | median travel past the broken edge, in ATR |
| `cont_0_5atr` / `cont_1atr` / `cont_1_5atr` (+ `_up` / `_dn`) | % travelling that far past the edge |
| `back_0_5atr` / `back_1atr` / `back_1_5atr` (+ `_up` / `_dn`) | % coming back that far inside |
| `back_to_mid`, `back_to_far_edge` (+ `_up` / `_dn`) | % back to the range's own levels |
| `width_med`, `width_atr_med` | range width in points and in ATRs |

With ~160 windows in the table, the top and bottom rows are partly the extremes
of a noisy distribution — the spread between best and worst is always wider than
the real effect. Use it to decide where to look, not what to trade.